In [ ]:
# CNV preprocessing for TCGA-BRCA  (LD modality, mirrors ADT pipeline)
# Run this ONCE before any OrchAMP survival notebook.
#
# Model: A_sel = L U^T + Z,  Z_j ~ iid N(0, 1)  (after per-feature normalisation)
#
# Steps:
#   1. Select top K_FEATURES CNV features by marginal variance
#   2. SVD of that submatrix; remove top K_SIGNAL signal PCs
#   3. Per-feature tau_j^2 from column-wise Frobenius norm of residual
#      (global tau^2 does NOT guarantee cov-I is PSD when noise is heteroscedastic)
#   4. Divide column j by tau_j  -->  noise ~ iid N(0,1) per feature
#   5. Verify cov(A_norm_centered) - I is PSD on full dataset
#   6. Save  (NOT mean-centered; pipeline mean-centers on train split only)
#
# Outputs (in ../matrices/):
#   CNV_X_ld_features.csv  -- (n_samples x K_FEATURES), ready for fit_lowdim
#
# In the OrchAMP notebooks, load this file and call:
#   pipe.fit_lowdim([X_cnv_ld_tr], top_features=None)
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
matrices_dir = Path("../matrices")

# Number of CNV features to keep as the low-dim modality
K_FEATURES = 5

# Number of signal PCs to remove when estimating tau^2
K_SIGNAL = 3

In [ ]:
# Load raw CNV
cnv_raw = pd.read_csv(matrices_dir / "CNV_X_full.csv", index_col=0)
A_full  = cnv_raw.to_numpy(dtype=np.float64)
n_samples, n_features = A_full.shape
print(f"Raw CNV: {n_samples} samples x {n_features} features")

In [ ]:
# Step 1: Feature selection -- top K_FEATURES by marginal variance (on raw data)
var_full = np.var(A_full, axis=0)
idx_top  = np.argpartition(var_full, -K_FEATURES)[-K_FEATURES:]
idx_top  = idx_top[np.argsort(var_full[idx_top])[::-1]]   # descending variance

A_sel = A_full[:, idx_top].copy()   # (n_samples, K_FEATURES)
print(f"Selected {K_FEATURES} features (indices): {idx_top}")
print(f"Their variances: {np.round(var_full[idx_top], 4)}")

In [ ]:
# Step 2-4: SVD of selected submatrix, estimate per-feature tau^2, normalize
# Global tau^2 (averaged over all features) does NOT guarantee cov(A_norm)-I is PSD
# when noise is heteroscedastic.  Per-feature tau^2 does: under A = L U^T + Z with
# Z_j ~ N(0, sigma_j^2), normalising by sigma_j gives A_norm = L_new U^T + Z_new
# where Z_new ~ iid N(0,1), so cov(A_norm) - I = Lnew Lnew^T >= 0 by construction.
U_sel, S_sel, Vt_sel = np.linalg.svd(A_sel, full_matrices=False)

k_sig = min(K_SIGNAL, K_FEATURES - 1)
A_sel_residual = A_sel - (U_sel[:, :k_sig] * S_sel[:k_sig]) @ Vt_sel[:k_sig, :]

# Per-feature residual variance
tau_sq_j = np.sum(A_sel_residual ** 2, axis=0) / n_samples   # shape (K_FEATURES,)
tau_j    = np.sqrt(tau_sq_j)

print(f"Top-{k_sig} singular values of selected submatrix: {np.round(S_sel[:k_sig], 3)}")
print(f"Per-feature tau : min={tau_j.min():.4f}  max={tau_j.max():.4f}  mean={tau_j.mean():.4f}")

A_norm = A_sel / tau_j[np.newaxis, :]   # noise ~ iid N(0,1) per feature

In [ ]:
# Step 5: Verify cov(A_norm_centered) - I is PSD on the full dataset
A_centered = A_norm - A_norm.mean(axis=0)
cov_A = (A_centered.T @ A_centered) / n_samples
vals  = np.linalg.eigvalsh(cov_A)   # real for symmetric matrix

print(f"Eigenvalues of cov(A_norm_centered)    : {np.round(np.sort(vals)[::-1], 4)}")
print(f"Eigenvalues of cov(A_norm_centered) - I: {np.round(np.sort(vals - 1)[::-1], 4)}")

n_neg = (vals - 1 < -1e-3).sum()
if n_neg == 0:
    print("PASS: cov - I is PSD -- L will be full-rank.")
else:
    print(f"WARNING: {n_neg} eigenvalue(s) of cov-I are < -1e-3.")
    print("Consider reducing K_FEATURES or increasing K_SIGNAL.")

In [ ]:
# Step 6: Save -- NOT mean-centered (pipeline centers on train split only)
out_path = matrices_dir / "CNV_X_ld_features.csv"
if out_path.exists():
    print(f"WARNING: {out_path.name} already exists -- overwriting.")

feature_names = cnv_raw.columns[idx_top]
cnv_ld = pd.DataFrame(A_norm, index=cnv_raw.index, columns=feature_names)
cnv_ld.to_csv(out_path)

print(f"Saved: {out_path.resolve()}")
print(f"Shape: {cnv_ld.shape}  (samples x K_FEATURES)")
print(f"In OrchAMP notebooks: fit_lowdim([X_cnv_ld_tr], top_features=None)")

In [ ]:
# Round-trip verification
check = pd.read_csv(out_path, index_col=0)
assert check.shape == cnv_ld.shape,              "Shape mismatch"
assert list(check.index) == list(cnv_raw.index), "Sample order changed"
assert np.allclose(check.values, A_norm, atol=1e-6), "Values differ after reload"
print("Round-trip verification passed.")